In [65]:
import numpy as np

# Read text

In [66]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [67]:
print(f'Characters: {len(text)}')

Characters: 1115393


In [68]:
print(f'{text[:100]}')

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [69]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f'All characters: {"".join(chars)}')
print(f'Vocab size: {vocab_size}')

All characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size: 65


# Character level tokenizer

In [70]:
encoded_dict = {chars[i]: i for i in range(vocab_size)}
decoded_dict = {i: chars[i] for i in range(vocab_size)}

def encode(s: str) -> list[int]:
    return [encoded_dict[c] for c in s]

def decode(s: list[int]) -> str:
    return "".join([decoded_dict[c] for c in s])

print(encode("test string"))
print(decode(encode("test string")))

[58, 43, 57, 58, 1, 57, 58, 56, 47, 52, 45]
test string


In [71]:
data = encode(text)
data = np.array(data, dtype=np.longfloat).reshape(-1, 1)

In [72]:
n = int(0.9*len(data))
X_train, X_test = data[:n], data[n:]
#print(f'n: {n}\nX_train: {X_train.shape}\nX_test: {X_test.shape}')

In [73]:
def create_sequences(data, seq_len = 8):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len, :])
        y.append(data[i+1:i+seq_len+1, :])
    return np.array(X), np.array(y)

X_train_seq, y_train_seq = create_sequences(X_train)
print(f'{X_train_seq.shape}, {y_train_seq.shape}')

(1003845, 8, 1), (1003845, 8, 1)


In [74]:
"""class Head(nn.Module):

    def __init__(self, n_embed, head_size, block_size, dropout=0.1):
        super().__init__()
        self.normalize_factor = head_size**0.5
        self.key = nn.Linear(n_embed, head_size)
        self.query = nn.Linear(n_embed, head_size)
        self.value = nn.Linear(n_embed, head_size)
        self.register_buffer('tril', torch.tril(torch.ones((block_size, block_size))))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        w = q @ k.transpose(-2, -1) / self.normalize_factor
        w = w.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # using mask makes it decoder block, in encoder every token can communicate
        w = F.softmax(w, dim=-1)
        w = self.dropout(w)

        output = w @ v
        return output"""

"class Head(nn.Module):\n\n    def __init__(self, n_embed, head_size, block_size, dropout=0.1):\n        super().__init__()\n        self.normalize_factor = head_size**0.5\n        self.key = nn.Linear(n_embed, head_size)\n        self.query = nn.Linear(n_embed, head_size)\n        self.value = nn.Linear(n_embed, head_size)\n        self.register_buffer('tril', torch.tril(torch.ones((block_size, block_size))))\n\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x):\n        B, T, C = x.shape\n\n        k = self.key(x)\n        q = self.query(x)\n        v = self.value(x)\n\n        w = q @ k.transpose(-2, -1) / self.normalize_factor\n        w = w.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # using mask makes it decoder block, in encoder every token can communicate\n        w = F.softmax(w, dim=-1)\n        w = self.dropout(w)\n\n        output = w @ v\n        return output"

In [75]:
from dlfs.layers import DenseLayer
from dlfs.activation import Softmax

class SingleAttentionHead():

    def __init__(self, n_embed, head_size, block_size, dropout=0.1):
        self.normalize_factor = head_size**0.5
        self.key = DenseLayer(n_embed, head_size)
        self.query = DenseLayer(n_embed, head_size)
        self.value = DenseLayer(n_embed, head_size)
        self.tril = np.tril(np.ones((block_size, block_size)))
        self.softmax = Softmax()
        #self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        self.key.forward(x)
        self.query.forward(x)
        self.value.forward(x)

        self.k = self.key.output
        self.q = self.query.output
        self.v = self.value.output
        
        self.w = np.matmul(self.q, self.k.swapaxes(-2, -1)) / self.normalize_factor
        mask_condition = self.tril[:T, :T] == 0
        self.w[:,mask_condition] = -np.inf
        self.softmax.forward(self.w)
        self.w = self.softmax.output
        self.output = np.matmul(self.w, self.v)
        #print(f'one head output: {self.output.shape}')

    def backward(self, delta):

        # Step 1: Gradient of the loss with respect to w (attention weights)
        d_w = np.matmul(delta, self.v.swapaxes(-2, -1))

        # Step 2: Gradient of the loss with respect to softmax input (logits)
        self.softmax.backward(d_w)  # Softmax backward pass
        d_w = self.softmax.dinputs

        # Step 3: Gradient of the loss with respect to w (before softmax)
        d_w = d_w * (self.w > 0).astype(float)  # Masking out invalid values from softmax

        # Step 4: Gradients w.r.t. key and query using the chain rule
        d_q = np.matmul(d_w, self.k)  # shape: (B, T, head_size)
        d_k = np.matmul(d_w.swapaxes(-2, -1), self.q)  # shape: (B, T, head_size)

        # Step 5: Update the key, query, and value parameters using the gradients
        # Gradient for the key (d_k) and query (d_q) go through the dense layers
        self.key.backward(d_k)
        self.query.backward(d_q)
        self.value.backward(np.matmul(d_w, self.v))

In [76]:
embedding = DenseLayer(1, 192)
head = SingleAttentionHead(n_embed=192, head_size=56, block_size=8)

x = X_train_seq[0]
x = x.reshape(1, *x.shape)
print(f'x {x.shape}')
embedding.forward(x)
print(f'embedding: {embedding.output.shape}')
head.forward(embedding.output)
print(f'output shape: {head.output.shape}')

x (1, 8, 1)
embedding: (1, 8, 192)
output shape: (1, 8, 56)


In [77]:
delta = np.random.rand(1, 8, 56)
head.backward(delta)
print(head.key.dinputs.shape, head.query.dinputs.shape, head.value.dinputs.shape)

(1, 8, 192) (1, 8, 192) (1, 8, 192)


In [78]:
class MultiHeadAttention():

    def __init__(self, n_embed, n_heads, head_size, block_size, dropout=0.1):
        self.n_heads = n_heads
        self.head_size = head_size

        # List to store each individual attention head
        self.attention_heads = [
            SingleAttentionHead(n_embed, head_size, block_size, dropout)
            for _ in range(n_heads)
        ]

        # Output Dense layer to combine the heads
        self.output_dense = DenseLayer(n_embed, n_embed)

        self.softmax = Softmax()

    def forward(self, x):

        # Store outputs of all attention heads
        head_outputs = []

        for head in self.attention_heads:
            head.forward(x)  # Compute attention for this head
            head_outputs.append(head.output)  # Store the output of each head

        # Concatenate the outputs of all heads along the last dimension (features)
        concatenated_output = np.concatenate(np.array(head_outputs), axis=-1)  # Shape: (B, T, n_heads * head_size)

        #print(f'concatenated: {concatenated_output.shape}')

        # Pass the concatenated output through the output dense layer
        self.output_dense.forward(concatenated_output)

        # Final output
        self.output = self.output_dense.output

    def backward(self, delta):
        # Step 1: Compute gradient w.r.t the output dense layer
        self.output_dense.backward(delta)
        d_concatenated_output = self.output_dense.output

        #print(f'd concat: {d_concatenated_output.shape}')

        # Step 2: Split the gradient back into the individual heads
        d_head_outputs = np.split(d_concatenated_output, self.n_heads, axis=-1)

        # Step 3: Backpropagate through each attention head
        for i, head in enumerate(self.attention_heads):
            head.backward(d_head_outputs[i])  # Backprop through each head


In [79]:
n_embed = 192
n_heads = 8
head_size = n_embed // n_heads

multihead = MultiHeadAttention(n_embed=n_embed, n_heads=n_heads, head_size=head_size, block_size=8)

embedding.forward(x)
multihead.forward(embedding.output)
print(f'x: {x.shape}')
print(f'embedding: {embedding.output.shape}')
print(f'multihead: {multihead.output.shape}')

x: (1, 8, 1)
embedding: (1, 8, 192)
multihead: (1, 8, 192)


In [80]:
delta = np.random.rand(1, 8, 192)
multihead.backward(delta)
for idx, head in enumerate(multihead.attention_heads):
    print(idx, head.key.dinputs.shape, head.query.dinputs.shape, head.value.dinputs.shape)

0 (1, 8, 192) (1, 8, 192) (1, 8, 192)
1 (1, 8, 192) (1, 8, 192) (1, 8, 192)
2 (1, 8, 192) (1, 8, 192) (1, 8, 192)
3 (1, 8, 192) (1, 8, 192) (1, 8, 192)
4 (1, 8, 192) (1, 8, 192) (1, 8, 192)
5 (1, 8, 192) (1, 8, 192) (1, 8, 192)
6 (1, 8, 192) (1, 8, 192) (1, 8, 192)
7 (1, 8, 192) (1, 8, 192) (1, 8, 192)


In [81]:
from dlfs.activation import ReLU

class FeedForward():

    def __init__(self, n_embed):
        self.fc1 = DenseLayer(n_embed, 4*n_embed)
        self.relu1 = ReLU()
        self.fc2 = DenseLayer(4*n_embed, n_embed)
        self.relu2 = ReLU()

    def forward(self, inputs):
        self.fc1.forward(inputs)
        self.relu1.forward(self.fc1.output)
        self.fc2.forward(self.relu1.output)
        self.relu2.forward(self.fc2.output)
        self.output = self.relu2.output

    def backward(self, delta):
        self.relu2.backward(delta)
        self.fc2.backward(self.relu2.dinputs)
        self.relu1.backward(self.fc2.dinputs)
        self.fc1.backward(self.relu1.dinputs)
        self.dinputs = self.fc1.dinputs

In [111]:
class LayerNorm:
    def __init__(self, num_features, epsilon=1e-5):
        """
        Initializes the LayerNorm layer.
        
        :param num_features: The number of features in the input (i.e., the dimension to normalize over).
        :param epsilon: Small value to prevent division by zero when computing the standard deviation.
        """
        self.num_features = num_features
        self.epsilon = epsilon
        
        # Initialize the scale (gamma) and shift (beta) parameters
        self.gamma = np.ones((1, num_features))  # Shape: (1, num_features)
        self.beta = np.zeros((1, num_features))  # Shape: (1, num_features)
        
    def forward(self, x):
        """
        Forward pass of LayerNorm
        
        :param x: Input data of shape (batch_size, num_features)
        :return: Layer normalized output
        """
        # Step 1: Compute mean and variance for each sample in the batch
        mean = np.mean(x, axis=1, keepdims=True)  # Shape: (batch_size, 1)
        self.variance = np.var(x, axis=1, keepdims=True)  # Shape: (batch_size, 1)
        
        # Step 2: Normalize the input
        self.x_normalized = (x - mean) / np.sqrt(self.variance + self.epsilon)  # Shape: (batch_size, num_features)
        
        # Step 3: Scale and shift the normalized values
        self.output = self.gamma * self.x_normalized + self.beta  # Shape: (batch_size, num_features)
    
    def backward(self, dout):
        """
        Backward pass for LayerNorm, computing the gradients.
        
        :param dout: The gradient of the loss with respect to the output.
        :return: Gradients with respect to input (dx), gamma, and beta.
        """
        # Step 1: Compute the gradient with respect to gamma and beta
        self.dgamma = np.sum(dout * self.x_normalized, axis=-2, keepdims=False)  # Gradient w.r.t. gamma
        self.dbeta = np.sum(dout, axis=-2, keepdims=False)  # Gradient w.r.t. beta
        
        # Step 2: Compute the gradient with respect to normalized x (dx_norm)
        dx_normalized = dout * self.gamma  # Shape: (batch_size, num_features)
        
        # Step 3: Compute the gradient with respect to x
        N = self.num_features  # Number of features (per example in the batch)
        self.dx = (1. / N) * (1. / np.sqrt(self.variance + self.epsilon)) * (
            N * dx_normalized - np.sum(dx_normalized, axis=1, keepdims=True) - self.x_normalized * np.sum(dx_normalized * self.x_normalized, axis=1, keepdims=True)
        )

In [112]:
data = np.random.rand(1, 8, 15)
ln = LayerNorm(15)
ln.forward(data)
print(ln.output.shape)

(1, 8, 15)


In [113]:
delta = np.random.rand(1, 8, 15)
ln.backward(delta)
print(ln.dgamma.shape, ln.dbeta.shape, ln.dx.shape)

(1, 15) (1, 15) (1, 8, 15)


In [98]:
print(ln.gamma.shape)

(1, 15)


In [ ]:
class Block:
    def __init__(self, n_embed, n_head, block_size):
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_heads=n_head, head_size=head_size, n_embed=n_embed, block_size=block_size)
        self.ffwd = FeedForward(n_embed)
        #self.ln1 = nn.LayerNorm(n_embed)
        #self.ln2 = nn.LayerNorm(n_embed)
    def forward(self, x):
        self.sa.forward(x)
        x = x + self.sa.output
        self.ffwd.forward(x)
        x = x + self.ffwd.output
        self.output = x

    def backward(self, delta):
        pass

In [83]:
n_embed = 192
n_heads = 8

b = Block(n_embed, n_heads, 8)

x = X_train_seq[0]
x = x.reshape(1, *x.shape)
embedding.forward(x)
b.forward(embedding.output)
print(f'x: {x.shape}')
print(f'embedding: {embedding.output.shape}')
print(f'block: {b.output.shape}')

x: (1, 8, 1)
embedding: (1, 8, 192)
block: (1, 8, 192)
